# Constraint-Based Rebate Contract Walkthrough

This notebook calls the production engine directly. It contains no duplicate optimization or payout formulas.

In [ ]:
from rebate_contract import (\n    AccountPeriodData, ScenarioSet, default_program_config,\n    generate_threshold_candidates, optimize_program,\n)\n\ndata = AccountPeriodData.from_csv('DummyDataGpot2.csv')\ndata.reconciliation()

## Compare the example grid with rounded data-supported candidates

The example is retained even when rejected, so its sparse cells and other failed guardrails remain visible.

In [ ]:
example = default_program_config()\ncandidates = [example, *generate_threshold_candidates(data, example)]\nresult = optimize_program(data, candidates, ScenarioSet(low=0.5, base=1.5, high=2.5))\n[(item.name, item.accepted, item.reasons[:2]) for item in result.candidate_assessments]

## Inspect the selected contract

Rows are growth milestones and columns are volume milestones, matching the published customer-facing grid.

In [ ]:
import pandas as pd\n\ngrid = pd.DataFrame(\n    result.rates,\n    index=[f'{value:.0%}+' for value in result.config.growth_milestones],\n    columns=[f'${value:,.0f}+' for value in result.config.volume_milestones],\n)\ndisplay(grid.style.format('{:.0%}'))\ndisplay(pd.DataFrame(result.scenarios).T)\nresult.constraint_checks, result.migration, result.exclusion_counts